# A실험 300장: 모델 5개 × 전처리 5개

**시작 전에:** 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 를 선택하세요.

위에서부터 순서대로 실행해요. 바꿀 곳은 **3번 셀의 zip 경로**뿐이에요.

| 순서 | 하는 일 | 걸리는 시간(예상) |
|---|---|---|
| 1~2 | 드라이브 연결, 코드 받기, 설치 | 3~5분 |
| 3~4 | zip 풀기, 채점표·검색 DB 만들기 (같은 시드라 매번 같은 분할) | 1분 |
| 5 | **시험 운전**: 5장만 돌려서 모델 5개가 다 켜지는지 확인 | 3~5분 |
| 6 | **1단계**: 개발용 × 25조합 | 1~2시간 |
| 7 | 1단계 채점 → 상위 4개 조합 | 1분 |
| 8 | **2단계**: 평가용 × 상위 4개 | 1~2시간 |
| 9 | 최종 채점 + 혼동행렬 | 1분 |

결과는 모두 드라이브 `exp300_results` 폴더에 바로 저장돼요. **끊기면 같은 셀을 다시 실행**하면 이어서 해요.

In [ ]:
# 1) 드라이브 연결 + 코드 최신으로 받기
from google.colab import drive
drive.mount('/content/drive')
import os
if os.path.exists('/content/dl_deep_imgps'):
    %cd /content/dl_deep_imgps
    !git pull -q
else:
    %cd /content
    !git clone -q https://github.com/yewon51019/dl_deep_imgps
    %cd /content/dl_deep_imgps
!nvidia-smi -L

In [ ]:
# 2) 설치 (Colab에 없는 것만)
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install pytesseract easyocr "python-doctr[torch]" pytorch-lightning timm nltk
# TrOCR(transformers)는 Colab에 이미 있어요

In [ ]:
# 3) ★ zip 경로 (준비 노트북에서 쓴 것과 같게)
ZIP_FILES = [
    '/content/drive/MyDrive/Colab Notebooks/pill_images_test/약제각인인식_테스트셋.zip',
    '/content/drive/MyDrive/Colab Notebooks/pill_images_test/약제각인인식_테스트셋_2.zip',   # ← 새로 받은 파일 이름으로
]
RESULT = '/content/drive/MyDrive/exp300_results'      # 결과 저장 폴더 (드라이브)
os.makedirs(RESULT, exist_ok=True)
for z in ZIP_FILES:
    print('있음 ✓' if os.path.exists(z) else '없음 ✗ (경로 확인!)', z)

In [ ]:
# 4) zip 풀기 + 채점표·검색 DB 만들기 (시드가 같아서 몇 번 해도 같은 분할)
import subprocess
# ※ 예전에 잘못된 zip을 풀어둔 적이 있으면 먼저:  !rm -rf /content/data1 /content/data2
ROOTS = []
for i, z in enumerate(ZIP_FILES, 1):
    out = f'/content/data{i}'
    if not os.path.exists(out):
        subprocess.run(['unzip', '-q', '-o', z, '-d', out], check=True)
    for cur, subdirs, _ in os.walk(out):
        if 'images' in subdirs and 'labels' in subdirs:
            ROOTS.append(cur)
print('찾은 폴더:', ROOTS)
MAN, DB = f'{RESULT}/manifest.csv', f'{RESULT}/search_db.csv'
!python exp300/make_manifest.py --roots {' '.join(repr(r) for r in ROOTS)} --out "{MAN}" --dev-ratio 0.2
!python exp300/make_search_db.py --label-roots {' '.join(repr(r + '/labels') for r in ROOTS)} --out "{DB}"

In [ ]:
# 5) 시험 운전: 개발용 5장 × 25조합. 모델마다 '[건너뜀]'이 안 나오는지 확인!
#    (처음엔 모델 파일을 내려받느라 느려요)
!python exp300/run_exp300.py --manifest "{MAN}" --db "{DB}" --split dev --limit 5 --out /content/pred_smoke.csv
!python exp300/evaluate_exp300.py --pred /content/pred_smoke.csv --out-dir /content/eval_smoke --top 4

In [ ]:
# 6) 1단계: 개발용 전부 × 25조합  (끊기면 이 셀만 다시 실행 → 이어서 해요)
!python exp300/run_exp300.py --manifest "{MAN}" --db "{DB}" --split dev --out "{RESULT}/pred_dev.csv"

In [ ]:
# 7) 1단계 채점: 25조합 순위표 + 상위 4개
!python exp300/evaluate_exp300.py --pred "{RESULT}/pred_dev.csv" --out-dir "{RESULT}/eval_dev" --top 4
TOP = open(f'{RESULT}/eval_dev/top_combos.txt').read().strip()
print('2단계에 쓸 조합:', TOP)

In [ ]:
# 8) 2단계: 평가용 × 상위 4개 조합  (끊기면 다시 실행)
#    ※ 여기서 나온 점수를 보고 조합/설정을 다시 고르면 안 돼요 (그러면 평가용이 개발용이 돼버림)
!python exp300/run_exp300.py --manifest "{MAN}" --db "{DB}" --split test --combos "{TOP}" --out "{RESULT}/pred_test.csv"

In [ ]:
# 9) 최종 채점 + 혼동행렬 (4개 조합 모두)
!python exp300/evaluate_exp300.py --pred "{RESULT}/pred_test.csv" --out-dir "{RESULT}/eval_test" --top 4 --cm all
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f'{RESULT}/eval_test/cm_*.png')):
    print(p); display(Image(p))

## 10. 각인 보정 실험 (OCR 다시 안 돌림, 몇 초)
- 부분일치 / 헷갈리는 글자(0↔O) / 후보 전체 사용 / 모델 합치기를 한 줄씩 쌓아서 효과를 봅니다.
- **방법은 개발 세트 점수로만 고르고**, 시험 세트는 그 방법을 한 번 적용해 최종 성적으로 씁니다.
- `pred_dev.csv` 가 Drive에 있어야 해요 (1단계 결과).

In [ ]:
!git pull -q
!python /content/dl_deep_imgps/exp300/correct_exp300.py \
    --dev {RESULT}/pred_dev.csv --test {RESULT}/pred_test.csv --db {RESULT}/search_db.csv \
    --base parseq:P0 --fuse parseq:P0,trocr:P3,easyocr:P0 --out {RESULT}/corrected